# <b> <font color='#A20025'> Appendix I &middot; Notebook 4 of 5: DiffSBDD: Designing Ligands Inside a Binding Pocket</font> </b>

*This is a bonus notebook accompanying Appendix I of [Build AI Drug Discovery Pipelines](https://github.com/nrflynn2/ml-drug-discovery). It is self-contained and runnable end to end; read it side by side with the appendix. This is the structure-based drug design capstone: you take an equivariant diffusion model off the shelf and design ligands inside the A2A binding pocket.*

In Notebook 3 you sampled 3D molecules from **FlowMol3**; floating freely in empty space, drawn only from the shape of drug-like chemical space. That is *ligand-based* generation: beautiful molecules, but with no idea of where they will bind.

Real drug discovery almost always starts from a **target**: a protein with a pocket you want to block. So the question changes from *"draw me a drug-like molecule"* to *"draw me a molecule that fits **this** pocket."* That is **structure-based drug design (SBDD)**, and it is what this notebook does with **DiffSBDD** (Schneuing et al., *Nature Computational Science* 2024); an **SE(3)-equivariant diffusion model** that generates a ligand *conditioned on a fixed protein pocket*.

You will not train anything. You **load one pretrained checkpoint** and use it for inference on CPU to:

1. **Generate** new ligands inside the pocket of a real target; the **adenosine A$_{2A}$ receptor** (PDB `3RFM`), a GPCR that caffeine binds.
2. **Profile** them; validity, drug-likeness (QED), synthesizability (SA), and **novelty** versus the reference ligand.
3. **Optimize** an existing molecule for higher QED; with the *same checkpoint*, no retraining. This is DiffSBDD's *"one model, many tasks"* selling point.

> **Why the A$_{2A}$ receptor?** The bundled example is `3RFM`: the thermostabilised adenosine A$_{2A}$ receptor crystallised **with caffeine bound**. So the reference ligand is a molecule you know intimately; a perfect yardstick for "is what DiffSBDD generates chemically reasonable, and is it *new*?"

<div style="border-left:5px solid #A20025;background:#fbeaed;padding:12px 16px;border-radius:4px;">
<b>Licensing; read before you run.</b><br>
DiffSBDD's <b>code</b> is MIT-licensed. The pretrained weights we download were trained on <b>CrossDocked2020</b>, whose curated splits are distributed under <b>GPL-2.0</b>. This notebook uses <b>only</b> (a) the pretrained checkpoint and (b) the tiny example PDBs bundled in the DiffSBDD repo. It <b>does not download or redistribute the CrossDocked dataset</b>. Using pretrained weights and small example structures is fine; redistributing the training dataset is not; so we never touch it.
</div>

## <b> <font color='#A20025'> Environment Setup</font> </b>

DiffSBDD needs a specific, **version-matched** stack. Three pins matter more than the rest, and each one will silently break the model if you get it wrong:

- **`pytorch-scatter` must match your exact PyTorch build.** It ships as a compiled extension keyed to a specific torch version *and* CPU-vs-CUDA flavour. Install a clean CPU torch wheel (`torch==2.0.1+cpu`) and the scatter wheel built against it, or you get an undefined-symbol error the moment you import.
- **OpenBabel comes from conda-forge, not pip.** DiffSBDD uses OpenBabel to infer bonds from the generated atom cloud. `pip install openbabel` compiles C++ and routinely fails; the conda-forge binary just works.
- **BioPython must be `< 1.80`.** DiffSBDD imports `three_to_one`, a function that was **removed in BioPython 1.80**. Pin `biopython==1.79`.

The cell below sets this up in a Colab-friendly way. On a fresh Colab CPU runtime it takes a few minutes the first time. This notebook is **CPU-only; no GPU required**.

In [ ]:
# --- Colab-friendly setup for DiffSBDD (CPU-only). Safe to re-run. ---
import importlib, subprocess, sys, os

def _have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    # 1) clean CPU PyTorch wheel (version string '2.0.1+cpu')
    if not _have('torch'):
        _pip('torch==2.0.1', '--index-url', 'https://download.pytorch.org/whl/cpu')
    # 2) torch-scatter built against torch 2.0.x CPU (note: PyG buckets 2.0.x
    #    under the torch-2.0.0+cpu path)
    if not _have('torch_scatter'):
        _pip('torch-scatter', '-f', 'https://data.pyg.org/whl/torch-2.0.0+cpu.html')
    # 3) model + chemistry deps
    for mod, pkg in [('pytorch_lightning','pytorch-lightning==1.8.4'),
                     ('rdkit','rdkit==2022.3.3'), ('Bio','biopython==1.79'),
                     ('py3Dmol','py3Dmol'), ('wandb','wandb==0.13.1')]:
        if not _have(mod):
            _pip(pkg)
    # 4) OpenBabel via conda (conda-forge binary; pip build routinely fails)
    if not _have('openbabel'):
        subprocess.run('pip install -q condacolab', shell=True)
        import condacolab; condacolab.install()   # restarts kernel; re-run cells after
    # 5) the DiffSBDD repo itself (MIT code)
    if not os.path.isdir('DiffSBDD'):
        subprocess.run('git clone -q https://github.com/arneschneuing/DiffSBDD.git', shell=True)
        os.chdir('DiffSBDD')

# CPU + offline hygiene
os.environ.setdefault('WANDB_MODE', 'offline')   # no experiment tracking at inference
os.environ.setdefault('OMP_NUM_THREADS', '4')
print('Setup ready; CPU-only, no GPU required.')

In [ ]:
# --- Path setup: resolve data/figures relative to this notebook, so paths work on Colab and locally ---
from pathlib import Path
import os

# On Colab, clone the repo and move into this notebook's folder on first run.
if not Path("data").exists() and not Path("../data").exists():
    if os.path.exists("/content"):  # Colab
        os.system("git clone --depth 1 https://github.com/nrflynn2/ml-drug-discovery.git")
        os.chdir("ml-drug-discovery/Appendix_I_FLYNN_ML4DD/notebooks")

HERE = Path.cwd()
APPENDIX = HERE.parent if HERE.name == "notebooks" else HERE
DATA = APPENDIX / "data"
FIGURES = APPENDIX / "figures"
ARTIFACTS = APPENDIX / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
print("data dir:", DATA)

### <b> <font color='#A20025'> Import Packages</font> </b>

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import time, json, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, Crippen, Lipinski, DataStructs, rdMolDescriptors
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

# DiffSBDD's synthetic-accessibility scorer ships in the repo (analysis/SA_Score)
sys.path.append(os.path.join('analysis', 'SA_Score'))
from analysis.SA_Score import sascorer

# the model class + repo helpers
from lightning_modules import LigandPocketDDPM
import utils

torch.set_num_threads(4)
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## <b> <font color='#A20025'> What pocket-conditioned diffusion actually does</font> </b>

In Notebook 2 you learned diffusion as a two-way street: a **forward** process that slowly adds Gaussian noise to data, and a learned **reverse** process that removes it, turning noise back into a sample. DiffSBDD keeps that machine, but changes two things; and both are the whole point of SBDD.

**Change 1; only the ligand is noised; the pocket is fixed context.** A complex is two sets of atoms: the **pocket** $z^{(P)}$ (protein atoms near the binding site) and the **ligand** $z^{(L)}$ (the molecule you want to design). The forward process adds noise to the *ligand only*:

$$q\!\left(z^{(L)}_t \mid z^{(L)}_{t-1}\right) = \mathcal{N}\!\left(z^{(L)}_t;\ \sqrt{1-\beta_t}\,z^{(L)}_{t-1},\ \beta_t I\right)$$

The pocket atoms never get noised. During the **reverse** process the network predicts how to denoise the ligand *while always seeing the clean pocket* as context, $p_\theta\!\left(z^{(L)}_{t-1} \mid z^{(L)}_t, z^{(P)}\right)$. So the pocket acts like a mold: noise flows into a ligand that fits it. (This is exactly the figure at the top of the DiffSBDD paper; noise on the left, a docked ligand on the right, the cyan pocket surface fixed the whole way.)

**Change 2; the network is SE(3)-equivariant.** A molecule in a pocket has no special origin or orientation; if you rotate and translate the whole complex, it is the *same* physical system. DiffSBDD's denoiser is built from an **E(n)-equivariant graph neural network (EGNN)** so that its predictions rotate and translate *with* the input. In one sentence:

> **Equivariance** means: rotate the pocket, and the generated ligand rotates the same way; the model never has to learn "the pocket, but tilted 30 degrees" as a separate case. This is why an equivariant model learns 3D binding geometry from far fewer examples than a model that treats coordinates as raw numbers.

Each reverse step denoises **both** the ligand's continuous **coordinates** and its discrete **atom types** (C, N, O, and so on) at once, so a finished sample is a full 3D molecule; geometry *and* chemistry; sitting in the pocket.

## <b> <font color='#A20025'> Load the checkpoint and point it at a pocket</font> </b>

We download the **conditional full-atom** CrossDocked checkpoint. *Conditional* is the key word: this model treats the pocket as fixed conditioning and generates only the ligand (the alternative *joint* models generate ligand and pocket together). The weights come from the authors' **Zenodo** record 8183747.

The example target is `example/3rfm.pdb`. We define the binding pocket the easy way: by pointing at the **reference ligand already in the crystal structure**; caffeine, in chain A at residue 330 (`A:330`). DiffSBDD takes every protein residue within 8 Å of that ligand as the pocket.

In [ ]:
CKPT = 'checkpoints/crossdocked_fullatom_cond.ckpt'
if not os.path.isfile(CKPT):
    os.makedirs('checkpoints', exist_ok=True)
    import urllib.request
    url = 'https://zenodo.org/record/8183747/files/crossdocked_fullatom_cond.ckpt'
    print('downloading checkpoint (~18 MB) ...')
    urllib.request.urlretrieve(url, CKPT)

t0 = time.time()
model = LigandPocketDDPM.load_from_checkpoint(CKPT, map_location='cpu').to('cpu').eval()
print(f'loaded checkpoint in {time.time()-t0:.1f}s')
print(f'  diffusion steps T = {model.T}   (we can sample with fewer at inference)')
n_params = sum(p.numel() for p in model.parameters())
print(f'  learnable params  = {n_params/1e6:.1f}M')

PDB = 'example/3rfm.pdb'          # adenosine A2A receptor
REF = 'A:330'                     # caffeine in chain A, residue 330 -> defines the pocket
print(f'\ntarget: {PDB}  |  pocket defined by reference ligand {REF} (caffeine)')

## <b> <font color='#A20025'> Generate ligands inside the pocket</font> </b>

One call does it. `generate_ligands` loads the PDB, cuts out the pocket around the reference ligand, samples how many atoms each new ligand should have, and runs the reverse diffusion; denoising ligand coordinates and atom types together while the pocket stays fixed.

Two knobs keep this **CPU-feasible**:

- **`n_samples`**; how many ligands to generate. We use a modest batch. Diffusion is many-step, so CPU generation runs at roughly **~15 s per molecule at 100 steps**.
- **`timesteps`**; denoising steps at inference. The model was trained with `T = 500`, but you can sample with fewer. We use **100**, which the chapter run showed is plenty for full validity here.

> On a GPU you would push `n_samples` into the hundreds. On CPU, start small; the recipe is identical, only the count changes.

In [ ]:
N_SAMPLES = 10          # chapter run used 20; 10 keeps CPU runtime ~2-3 min
TIMESTEPS = 100         # denoising steps at inference (out of T=500)

torch.manual_seed(2024)
t0 = time.time()
raw_mols = model.generate_ligands(
    PDB, N_SAMPLES, pocket_ids=None, ref_ligand=REF,
    num_nodes_lig=None, sanitize=False, largest_frag=True,
    relax_iter=0, timesteps=TIMESTEPS)
dt = time.time() - t0
print(f'generated {len(raw_mols)} ligands in {dt:.0f}s  ({dt/max(1,len(raw_mols)):.1f}s/mol)')
print('atom counts:', [m.GetNumAtoms() if m else 0 for m in raw_mols])

## <b> <font color='#A20025'> Are they real molecules? Validity, then SDF</font> </b>

A generated atom cloud only counts as a *molecule* if RDKit can **sanitize** it; valences satisfied, rings kekulizable, a chemically consistent graph. Following the same convention we used for FlowMol3, we take each molecule's **largest fragment** and call it valid if it sanitizes.

Valid molecules are written to **`diffsbdd_generated.sdf`** *with their 3D coordinates in the pocket frame*; so you can open them in any viewer, or hand one to a docking tool.

In [ ]:
def sanitized_largest_frag(m):
    if m is None: return None, 0
    try:
        frags = Chem.GetMolFrags(m, asMols=True, sanitizeFrags=False)
    except Exception:
        return None, 0
    if not frags: return None, 0
    largest = max(frags, key=lambda f: f.GetNumAtoms())
    try:
        mm = Chem.Mol(largest); Chem.SanitizeMol(mm); return mm, len(frags)
    except Exception:
        return None, len(frags)

valid_mols, n_conn = [], 0
w = Chem.SDWriter('diffsbdd_generated.sdf'); w.SetKekulize(False)
for i, raw in enumerate(raw_mols):
    san, nf = sanitized_largest_frag(raw)
    if san is None: continue
    valid_mols.append(san)
    if nf == 1: n_conn += 1
    raw.SetProp('_Name', f'diffsbdd_3rfm_{i}')
    raw.SetProp('SMILES', Chem.MolToSmiles(san))
    w.write(raw)
w.close()

nv = len(valid_mols)
print(f'validity : {nv}/{N_SAMPLES} = {100*nv/N_SAMPLES:.0f}%')
print(f'connected: {n_conn}/{N_SAMPLES} single-fragment')
print('wrote diffsbdd_generated.sdf')

In the chapter run (20 samples at 100 steps) DiffSBDD reached **100% graph validity, 100% single-fragment** on this pocket. Your smaller batch will land close. Be precise about what that measures, though, because the wording matters more here than almost anywhere else in this appendix: every generated atom cloud was a **chemically legal, connected molecular graph**. Valences add up, rings kekulize, nothing came back as a scatter of fragments. It is a real result, and the equivariant 3D prior is doing real work to earn it.

It is also silent about geometry. RDKit sanitization never inspects a coordinate, so a molecule can pass it while its bond angles are bent well outside anything a real compound adopts. On the physical bar, which we measure a few cells from now, **15 of these 20 ligands pass the full protein-aware PoseBusters suite** and their median UFF strain energy is **134.48 kcal/mol against the 0.388 of the caffeine co-crystal pose**. Both facts describe the same 20 molecules. Hold the 100% and the 15/20 together; the gap between them is the most useful thing this notebook has to teach about generative chemistry metrics.

## <b> <font color='#A20025'> The physical bar: does the geometry survive inspection?</font> </b>

The 100% above is a statement about the molecular **graph**. It says valences add up, rings kekulize,
and every atom belongs to one connected component. It says nothing whatsoever about where those atoms
are in space, because RDKit sanitization never looks at a coordinate.

For a pocket-conditioned generator that is the wrong bar, and we should say so plainly. DiffSBDD's entire
selling point is that it produces a ligand **already seated in the binding site**, in the pocket's own
coordinate frame. If the coordinates are not physically defensible, the placement is not either, and the
graph-level 100% will never tell us. So we are going to measure the geometry directly, with two tools the
structure-based design field uses for exactly this purpose, and report what we find rather than what we
hoped for. The short version, before we start: the ligands are placed correctly with respect to the
protein and internally strained, which is a more interesting result than either "it works" or "it does not".

**PoseBusters** (Buttenschoen, Morris and Deane) is a battery of independent physical tests, each
returning a plain pass or fail. It exists because deep-learning docking methods were shown to report
sub-2 Angstrom RMSD to a crystal pose while handing back ligands with impossible bond geometry or atoms
overlapping the protein; RMSD alone could not catch that, so the field needed a physical checklist. Its
checks come in two families, and the distinction is the whole story here. **Intramolecular** checks look
only at the ligand: bond lengths and angles against reference distributions, internal steric clash,
ring flatness and non-flatness, double-bond planarity, and a UFF energy ratio against a relaxed
conformer ensemble. **Intermolecular** checks look at the ligand against the receptor: minimum distance
to protein, to organic and inorganic cofactors and to waters, volume overlap with each of those, and a
maximum-distance check that the ligand has not wandered out of the site.

Those intermolecular checks are why the receptor we hand PoseBusters has to be the **full 3RFM protein**,
not the pocket. The `diffsbdd_complex.pdb` we saved earlier contains a 36-residue pocket shell, which is
exactly right for a picture and useless here: "minimum distance to protein" computed against 36 residues
cannot detect a ligand atom buried in the 37th, and "volume overlap with protein" against a shell that
surrounds the ligand by construction will always look clean. We rebuild the receptor from all 2,250 ATOM
records of the PDB the repo ships, so the checks mean what their names say.

<div style="border-left:5px solid #A20025;background:#fbeaed;padding:12px 16px;border-radius:4px;">
<b>Caution: two PoseBusters settings that will silently give you the wrong answer.</b><br>
<b>Use <code>config='dock'</code>, not <code>'redock'</code>.</b> The <code>redock</code> configuration adds six checks
that compare your pose to a known correct one: molecular formula, molecular bonds, double-bond
stereochemistry, tetrahedral chirality, and RMSD within 2 Angstrom, plus the loading check for the
reference itself. All of them need a <code>mol_true</code> reference pose. De novo ligands have no reference pose
by definition, so on this batch those checks cannot pass; <code>dock</code> is the configuration built for the case
where you have a receptor but no answer key.<br><br>
<b>Leave <code>full_report=False</code>.</b> With <code>full_report=True</code> PoseBusters returns 79 boolean columns
instead of 22, and several of them are <code>False</code> for a structurally uninteresting reason: <code>mol_true_loaded</code>
is <code>False</code> because there is no reference, and the cofactor and water columns are <code>False</code> or empty because
3RFM has neither in the file we built. Take a naive conjunction over "all boolean columns" and you get
<b>0 of 20 valid</b>, a number that is pure artifact. Score the checks the configuration chose, which is
what the cell below does.
</div>

In [ ]:
# PoseBusters against the FULL 3RFM receptor. USE_CACHED = True reads the committed result.
# Set it to False to recompute: about 130 s for 20 ligands on a laptop CPU, and it needs
# `pip install posebusters` (not part of the DiffSBDD stack installed above).
import io
import pandas as pd

USE_CACHED = True

# Every metrics CSV in this appendix carries a provenance header as leading '#' lines. Strip
# ONLY those leading lines by hand. Do NOT pass comment='#' to read_csv: a '#' inside a SMILES
# string is a triple bond (C#N is a nitrile), and pandas would truncate that field mid-value
# and silently leave NaN behind. Two of the 20 SMILES in this batch are nitriles, so passing
# comment='#' really does corrupt this file rather than merely risking it.
def read_metrics_csv(path):
    lines = open(path).read().splitlines()
    i = 0
    while i < len(lines) and lines[i].lstrip().startswith("#"):
        i += 1
    return pd.read_csv(io.StringIO("\n".join(lines[i:])))


PB_CSV = DATA / "posebusters_diffsbdd.csv"
if not PB_CSV.exists():
    PB_CSV = Path("posebusters_diffsbdd.csv")       # fallback: local working copy

INTRA_CHECKS = ['sanitization', 'inchi_convertible', 'all_atoms_connected', 'no_radicals',
                'bond_lengths', 'bond_angles', 'internal_steric_clash', 'aromatic_ring_flatness',
                'non-aromatic_ring_non-flatness', 'double_bond_flatness', 'internal_energy']
INTER_CHECKS = ['protein-ligand_maximum_distance', 'minimum_distance_to_protein',
                'minimum_distance_to_organic_cofactors', 'minimum_distance_to_inorganic_cofactors',
                'minimum_distance_to_waters', 'volume_overlap_with_protein',
                'volume_overlap_with_organic_cofactors', 'volume_overlap_with_inorganic_cofactors',
                'volume_overlap_with_waters']

if USE_CACHED:
    pb = read_metrics_csv(PB_CSV)
else:
    from posebusters import PoseBusters
    # The receptor: all ATOM records of the PDB the repo ships, and only those. Dropping the
    # HETATM records is deliberate. The bound caffeine is one of them, and leaving it in makes
    # PoseBusters treat the reference ligand as an organic cofactor that every generated ligand
    # overlaps by construction, which fails minimum_distance_to_organic_cofactors and
    # volume_overlap_with_organic_cofactors for all 20 and drops the verdict to 0 valid.
    RECEPTOR = 'diffsbdd_receptor_full.pdb'
    with open(RECEPTOR, 'w') as fh:
        for line in open(PDB):
            if line.startswith('ATOM'):
                fh.write(line)
        fh.write('END\n')

    pb_raw = PoseBusters(config='dock').bust(          # 'dock', never 'redock'; see the Caution
        mol_pred=valid_mols, mol_cond=RECEPTOR, full_report=False)
    pb = pb_raw.reset_index()[INTRA_CHECKS + INTER_CHECKS]

# A check returns NaN when it cannot be evaluated at all: UFF has no parameters for two of these
# ligands, so their energy ratio is undefined rather than bad. Report both readings.
checks = pb[INTRA_CHECKS + INTER_CHECKS].astype('boolean')
n_mols = len(pb)
n_pass = int(checks.apply(lambda row: row.dropna().all(), axis=1).sum())
n_pass_strict = int(checks.fillna(False).all(axis=1).sum())

print(f'PoseBusters-valid (all evaluated checks pass): {n_pass}/{n_mols} = {100*n_pass/n_mols:.0f}%')
print(f'PoseBusters-valid (unevaluable counted as failure): {n_pass_strict}/{n_mols}'
      f' = {100*n_pass_strict/n_mols:.0f}%')
print(f'checks that could not be evaluated: {int(checks.isna().sum().sum())}'
      f' (UFF parameters missing)\n')

print('INTRAMOLECULAR (the ligand on its own):')
for check in INTRA_CHECKS:
    print(f'  {check:<32s} {100*float(checks[check].mean()):5.1f}%')
print('\nINTERMOLECULAR (the ligand against the receptor):')
for check in INTER_CHECKS:
    print(f'  {check:<40s} {100*float(checks[check].mean()):5.1f}%')

Read the two blocks against each other, because the contrast is the finding.

**Every intermolecular check passes for all 20 ligands.** Minimum distance to protein, volume overlap
with protein, the maximum-distance check: 100% across the board. DiffSBDD is doing the job it was built
for. The ligands sit in the pocket without clipping into the receptor and without drifting out of the
site, and that is a real result from a model that was given nothing but a protein structure.

**The intramolecular checks are where it comes apart.** Bond angles pass for only 85% of the batch, bond
lengths for 90%, non-aromatic ring non-flatness for 95%. Fifteen of 20 ligands pass everything that could
be evaluated; 13 of 20 if we count the two whose UFF energy ratio could not be computed as failures
rather than as missing data. Every single failure is in the intramolecular family.

So the sentence to carry forward is precise: DiffSBDD places molecules cleanly **relative to the
protein** while distorting their **internal geometry**. That is a coherent picture of what the model
learned. Pocket conditioning trains the network hard on the ligand-protein relationship, which is what
the loss can see; local bond geometry has to be picked up incidentally from the training distribution,
and one in four of these ligands did not pick up enough of it. Note also that 100% graph validity and
75% physical validity are both true of the same 20 molecules. They are not in conflict; they are
answers to different questions, and only one of them is about 3D structure.

In [ ]:
# PoseCheck: strain energy, clash count, and the interaction profile, with the caffeine
# co-crystal pose as the reference row. Strain is the sharpest number in this notebook and it
# is not a pass/fail check at all, which is precisely why PoseBusters cannot surface it.
#   strain = the pose's UFF energy minus the energy of a relaxed conformer ensemble, kcal/mol.
# Fresh compute needs `pip install posecheck` and about 2.4 s per ligand; PoseCheck protonates
# the receptor internally with `hydride`, so no manual preparation is required.
USE_CACHED = True

PC_CSV = DATA / "posecheck_diffsbdd.csv"
if not PC_CSV.exists():
    PC_CSV = Path("posecheck_diffsbdd.csv")

if USE_CACHED:
    pc_df = read_metrics_csv(PC_CSV)
else:
    from posecheck import PoseCheck
    pc = PoseCheck()
    pc.load_protein_from_pdb('diffsbdd_receptor_full.pdb')   # written by the cell above
    pc.load_ligands_from_mols(valid_mols + [ref_mol])        # caffeine last, as the reference
    # Expect your strain numbers to land near ours rather than exactly on them. PoseCheck builds
    # the relaxed reference ensemble with AllChem.EmbedMolecule and passes no random seed, so
    # the ensemble minimum drifts between runs; recomputing this batch moved individual ligands
    # by up to ~3% and left most of them bit-identical. Clash counts and interactions are
    # deterministic. The conclusions here rest on orders of magnitude, not on the last digit.
    strain_vals = pc.calculate_strain_energy()
    clash_vals = pc.calculate_clashes()
    interactions = pc.calculate_interactions()
    # calculate_interactions returns a prolif MultiIndex frame keyed
    # (ligand, residue, interaction_type); count how many of each type fired per ligand.
    rows = []
    for k in range(len(strain_vals)):
        per_type = {}
        for (_lig, _res, itype), fired in interactions.iloc[k].items():
            if bool(fired) if pd.notna(fired) else False:
                per_type[itype] = per_type.get(itype, 0) + 1
        rows.append({'strain_energy_kcal_mol': strain_vals[k], 'clash_count': clash_vals[k],
                     'n_hbond_donors': per_type.get('HBDonor', 0),
                     'n_hbond_acceptors': per_type.get('HBAcceptor', 0),
                     'n_hydrophobic': per_type.get('Hydrophobic', 0),
                     'n_vdw_contacts': per_type.get('VdWContact', 0),
                     'is_reference': k == len(strain_vals) - 1})
    pc_df = pd.DataFrame(rows)

gen = pc_df[~pc_df['is_reference'].astype(bool)]
ref = pc_df[pc_df['is_reference'].astype(bool)]
caffeine_strain = float(ref['strain_energy_kcal_mol'].iloc[0])

print(f'UFF strain energy, kcal/mol   ({len(gen)} generated ligands + 1 co-crystal reference)')
print(f'  generated: median {gen["strain_energy_kcal_mol"].median():.2f}'
      f'   IQR [{gen["strain_energy_kcal_mol"].quantile(0.25):.2f},'
      f' {gen["strain_energy_kcal_mol"].quantile(0.75):.2f}]'
      f'   max {gen["strain_energy_kcal_mol"].max():.1f}')
print(f'  caffeine co-crystal pose: {caffeine_strain:.3f}')
print(f'  median fold over the reference: {gen["strain_energy_kcal_mol"].median()/caffeine_strain:.0f}x')
print(f'  ligands above 100 kcal/mol: {int((gen["strain_energy_kcal_mol"] > 100).sum())}/{len(gen)}'
      f'   above 300: {int((gen["strain_energy_kcal_mol"] > 300).sum())}/{len(gen)}')

print(f'\nClash count (PoseCheck default 0.5 A tolerance)')
print(f'  generated: median {gen["clash_count"].median():.1f}'
      f'   IQR [{gen["clash_count"].quantile(0.25):.0f}, {gen["clash_count"].quantile(0.75):.0f}]'
      f'   max {int(gen["clash_count"].max())}'
      f'   zero-clash ligands: {int((gen["clash_count"] == 0).sum())}')
print(f'  caffeine co-crystal pose: {int(ref["clash_count"].iloc[0])}'
      '   <-- ABOVE the generated median')

print('\nProtein-ligand interactions detected by prolif')
print(f'{"":<22s}{"median":>8s}{"mean":>7s}{"max":>6s}{"caffeine":>10s}')
for col, label in [('n_hbond_donors', 'H-bond donors'), ('n_hbond_acceptors', 'H-bond acceptors'),
                   ('n_hydrophobic', 'hydrophobic'), ('n_vdw_contacts', 'van der Waals')]:
    print(f'  {label:<20s}{gen[col].median():>8.1f}{gen[col].mean():>7.2f}'
          f'{int(gen[col].max()):>6d}{int(ref[col].iloc[0]):>10d}')

Three results here, and the first one is the reason this section exists.

**Strain is where the geometry problem shows up quantitatively.** The generated ligands have a median UFF
strain of 134.48 kcal/mol against the caffeine co-crystal pose's 0.388, a factor of 347. All 20 exceed
the reference, 14 exceed 100 kcal/mol, 5 exceed 300, and the worst reaches 2601. For calibration,
thermal energy at body temperature is well under 1 kcal/mol per degree of freedom, so a few hundred
kcal/mol of strain does not describe a conformer a molecule would ever visit in solution. And note what
this means for our metrics: strain is invisible to the 100% validity figure, and it is invisible to
PoseBusters too, whose energy-ratio check passed for all 18 ligands it could evaluate. It only appears
when you measure a continuous quantity instead of asking a pass or fail question. Compare FlowMol3's
median of 8.18 from Notebook 3: same measurement, same reference, roughly sixteen times less strain from
the generator that was not conditioned on a pocket at all.

**Clash count does not discriminate, so do not cite it.** The generated median is 2.5 clashes, and
caffeine's own crystallographic pose scores 3. A number that ranks an experimentally determined structure
below the median of a set of generated ones is not measuring quality; at a 0.5 Angstrom tolerance it is
mostly counting ordinary van der Waals contact. Two generated ligands score zero clashes, which is
tempting to quote and would be meaningless. Report strain, which separates the batch from the reference by
two orders of magnitude, and let clash count go.

**The interaction profile is thin, and honestly reported.** Median zero hydrogen-bond acceptors, two
hydrophobic contacts, four and a half van der Waals contacts per ligand. Caffeine, for comparison, makes
one acceptor contact, no hydrophobic contacts, and five van der Waals contacts, so the generated ligands
are not obviously worse than the reference on this axis; there is simply not much specific interaction
chemistry happening in either case. One column deserves an explicit note because a reader will wonder
whether it is broken: **H-bond donors is zero for all 21 ligands, caffeine included.** That is a measured
zero, not a missing column. Donor detection was requested from prolif and never fired, which is correct
for caffeine (it has no donor hydrogen), and the generated set happens to place none either. Anionic,
cationic, cation-pi and pi-stacking interactions were likewise requested and never detected. The types
that did fire are HBAcceptor, Hydrophobic and VdWContact.

![Figure I.7b]({{artifact:56ef2f07-1a33-4ea7-8e87-a29b82d2216b}})

**Figure I.7b.** Physical plausibility for both generators in this appendix, on the three axes that
graph validity cannot see. Panel (a) gives the eleven intramolecular PoseBusters checks as the percentage
of molecules passing, DiffSBDD in crimson against FlowMol3 from Notebook 3 in slate: DiffSBDD drops to
90% on bond lengths, 85% on bond angles and 95% on non-aromatic ring non-flatness, and its energy-ratio
check could only be evaluated for 18 of its 20 ligands, while FlowMol3 clears all eleven checks at 100%.
Panel (b) shows the nine ligand-to-protein checks, which exist only for DiffSBDD because FlowMol3 samples
into empty space with no receptor; every one passes for all 20 ligands, so the pocket-conditioned model
places its molecules correctly with respect to the protein it was given. Panel (c) plots UFF strain
energy on a log scale, one point per molecule, with the caffeine co-crystal pose as the dashed reference
at 0.39 kcal/mol: DiffSBDD's median of 134 sits two orders of magnitude above the crystal pose and one
above FlowMol3's median of 8, with its worst ligand at 2601. Together the panels locate the problem
precisely. It is intramolecular geometry rather than placement, and it is invisible to both the 100%
validity figure and to PoseBusters' pass or fail battery.

## <b> <font color='#A20025'> Profile them: drug-likeness, synthesizability, novelty</font> </b>

Validity says the graph is legal; it doesn't say the molecule is *good* or *new*. Three medicinal-chemistry lenses:

- **QED** (drug-likeness), $0\to1$, higher is better.
- **SA score** (synthetic accessibility), $1\to10$, **lower = easier** to make.
- **Similarity to the reference ligand**; the Tanimoto similarity of each generated molecule to **caffeine**, the ligand that defined this pocket. Low values mean DiffSBDD did not hand back a copy of the molecule it was pointed at. This is often labelled *novelty*, and we will not call it that: novelty is a claim about the training set, not about one reference ligand, and a later section measures both so you can see how far apart they are.

We compute all three for every valid molecule.

In [ ]:
ref_mol = Chem.SDMolSupplier('example/3rfm_B_CFF.sdf', sanitize=True)[0]  # caffeine
ref_fp  = Chem.RDKFingerprint(ref_mol)
print('reference (caffeine):', Chem.MolToSmiles(ref_mol), '| QED', round(QED.qed(ref_mol),2))

qed = np.array([QED.qed(m) for m in valid_mols])
sa  = np.array([sascorer.calculateScore(m) for m in valid_mols])
tan = np.array([DataStructs.TanimotoSimilarity(Chem.RDKFingerprint(m), ref_fp) for m in valid_mols])
print(f'QED      : median {np.median(qed):.2f}  range [{qed.min():.2f}, {qed.max():.2f}]')
print(f'SA score : median {np.median(sa):.2f}  range [{sa.min():.2f}, {sa.max():.2f}]')
print(f'novelty  : median Tanimoto-to-caffeine {np.median(tan):.2f}  (max {tan.max():.2f})')

![Figure I.7]({{artifact:356bfd72-a6c5-4c3e-80e8-ad19124380a4}})

**Figure I.7.** The generated ligands profiled against caffeine, the pocket's native reference. Each generated molecule is one point across the QED, synthetic-accessibility, molecular-weight and similarity panels, with caffeine drawn as a dashed reference line. The batch spreads around drug-like values (median QED 0.60) while staying chemically distinct from caffeine (median Tanimoto 0.21), showing DiffSBDD samples novel occupants of the pocket rather than copies of the known ligand.

The chapter figure below profiles the full 20-molecule run. Each point is one generated ligand; the bar is the median; the dashed line is caffeine as a yardstick. The story reads straight off the panels: the generated ligands are **as drug-like as caffeine or better** (median QED 0.60 vs caffeine 0.54), **comfortably synthesizable**, and **all far from caffeine** (every Tanimoto below 0.4). Read that third panel as distance from the reference ligand rather than as novelty; measured against a sample of the training corpus instead, the median nearest-neighbour similarity is 0.4664 under the same fingerprint.

In [ ]:
# The chapter renders this with the shared figure style; here is the plain version.
fig, ax = plt.subplots(1, 3, figsize=(10, 3.2))
rng = np.random.default_rng(7)
for a, vals, lab, ref, ylim in [
    (ax[0], qed, 'QED (higher = better)', QED.qed(ref_mol), (0,1)),
    (ax[1], sa,  'SA score (lower = easier)', sascorer.calculateScore(ref_mol), (1,8)),
    (ax[2], tan, 'Tanimoto to caffeine', 1.0, (0,1))]:
    x = 1 + rng.uniform(-0.12, 0.12, size=len(vals))
    a.scatter(x, vals, s=32, color='#A20025', alpha=0.8, edgecolor='white')
    a.plot([0.8,1.2], [np.median(vals)]*2, color='#A20025', lw=2.5)
    a.axhline(ref, color='#333', ls='--', lw=1.2)
    a.set_xticks([]); a.set_ylabel(lab); a.set_ylim(*ylim)
ax[0].set_title('Drug-likeness'); ax[1].set_title('Synthetic accessibility'); ax[2].set_title('Novelty')
fig.suptitle('DiffSBDD ligands for the adenosine A$_{2A}$ receptor (3RFM)', y=1.03)
fig.tight_layout(); plt.show()

*(The cell above renders the figure inline when you run it. The chapter ships a polished version; `diffsbdd_properties.png`; drawn from the same QED / SA / Tanimoto arrays, with caffeine marked as the reference on every panel. The full per-molecule table; QED, SA, logP, MW, Lipinski count, Tanimoto-to-caffeine; lives in `diffsbdd_metrics.csv`.)*

## <b> <font color='#A20025'> Novel compared to what? Naming the metric</font> </b>

The 0.21 above is real and it reproduces exactly. What it is **not** is novelty. It is the median
Tanimoto similarity between each generated ligand and **caffeine**, the one reference ligand that defined
the pocket. A low value there tells us the model did not hand back a copy of the molecule it was pointed
at, which is worth knowing and is not the question the word "novelty" raises. The question a reader
actually has is whether the model is reciting its training set, and caffeine is a sample of size one from
a training corpus of hundreds of thousands. Those are different measurements. This section computes both,
side by side, under both fingerprints, so the notebook is the source of every number.

Definitions first, because a similarity without them is not reproducible. A **fingerprint** is a
fixed-length bit vector summarizing which substructures a molecule contains. **Tanimoto similarity** is
the number of bits two fingerprints share over the number set in either, from 0 (nothing in common) to 1
(bit-identical). We use two fingerprints, and they are not interchangeable. `Chem.RDKFingerprint` at
RDKit's defaults is **path-based**: it hashes linear atom paths of length 1 through 7 into 2048 bits with
2 bits per hash, and radius is not one of its parameters. `Morgan r=2/2048` is **circular** (the ECFP4
family): it hashes each atom's environment out to two bonds into 2048 bits. The cell below prints every
number under both, which is the only honest way to present a similarity.

In [ ]:
# Novelty measured two ways, under two fingerprints. USE_CACHED = True reads the committed
# per-ligand table. Set it to False to recompute against the sampled CrossDocked2020 corpus
# that ships alongside it (a couple of seconds: 20 ligands against 5,211 corpus molecules).
USE_CACHED = True

NOV_CSV = DATA / "novelty_diffsbdd.csv"
if not NOV_CSV.exists():
    NOV_CSV = Path("novelty_diffsbdd.csv")
CORPUS_SMI = DATA / "corpus_sample_crossdocked.smi"
if not CORPUS_SMI.exists():
    CORPUS_SMI = Path("corpus_sample_crossdocked.smi")

from rdkit.Chem import rdFingerprintGenerator
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def both_fingerprints(mol):
    """Path-based RDKit fingerprint at defaults, and circular Morgan r=2. Both 2048 bits."""
    return Chem.RDKFingerprint(mol), morgan_gen.GetFingerprint(mol)


def read_corpus_smi(path):
    """Tab-separated SMILES with a leading '#' provenance header. Column 1 is a flag marking
    ligands independently confirmed present in DiffSBDD's own published train split."""
    lines = open(path).read().splitlines()
    i = 0
    while i < len(lines) and lines[i].lstrip().startswith("#"):
        i += 1
    smiles, confirmed = [], []
    for line in lines[i:]:
        if not line.strip():
            continue
        parts = line.split('\t')
        smiles.append(parts[0])
        confirmed.append(len(parts) > 1 and parts[1] == '1')
    return smiles, confirmed


if USE_CACHED:
    nov = read_metrics_csv(NOV_CSV)
else:
    corpus_smiles, in_train_split = read_corpus_smi(CORPUS_SMI)
    corpus, corpus_confirmed = [], []
    for smi, flag in zip(corpus_smiles, in_train_split):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        corpus.append(both_fingerprints(mol))
        corpus_confirmed.append(flag)
    corpus_rdkit = [a for a, _ in corpus]
    corpus_morgan = [b for _, b in corpus]
    confirmed_rdkit = [a for (a, _), f in zip(corpus, corpus_confirmed) if f]
    confirmed_morgan = [b for (_, b), f in zip(corpus, corpus_confirmed) if f]
    print(f'corpus: {len(corpus_rdkit)} unique ligands, of which '
          f'{len(confirmed_rdkit)} are confirmed in the published train split')

    caff_rdkit, caff_morgan = both_fingerprints(Chem.RemoveHs(ref_mol))   # caffeine, from 3RFM
    rows = []
    for mol in [Chem.RemoveHs(m) for m in valid_mols]:
        fp_rdkit, fp_morgan = both_fingerprints(mol)
        rows.append({
            'smiles': Chem.MolToSmiles(mol),
            'max_tanimoto_vs_train_rdkit': max(DataStructs.BulkTanimotoSimilarity(fp_rdkit, corpus_rdkit)),
            'max_tanimoto_vs_train_morgan': max(DataStructs.BulkTanimotoSimilarity(fp_morgan, corpus_morgan)),
            'max_tanimoto_vs_trainsplit_confirmed_rdkit':
                max(DataStructs.BulkTanimotoSimilarity(fp_rdkit, confirmed_rdkit)),
            'max_tanimoto_vs_trainsplit_confirmed_morgan':
                max(DataStructs.BulkTanimotoSimilarity(fp_morgan, confirmed_morgan)),
            'tanimoto_vs_caffeine_rdkit': DataStructs.TanimotoSimilarity(fp_rdkit, caff_rdkit),
            'tanimoto_vs_caffeine_morgan': DataStructs.TanimotoSimilarity(fp_morgan, caff_morgan)})
    nov = pd.DataFrame(rows)

# Per-ligand MAXIMUM Tanimoto to the corpus (its nearest training neighbour), then the MEDIAN
# of those maxima. That is the standard 'is it memorising' statistic; the caffeine column is
# the chapter's published 'novelty' number, kept here for direct comparison.
print(f'{len(nov)} ligands  |  median Tanimoto, higher = more similar\n')
print(f'{"":<48s}{"RDKit path-based":>18s}{"Morgan r=2":>13s}')
for label, rd, mg in [
        ('nearest neighbour, sampled training corpus',
         'max_tanimoto_vs_train_rdkit', 'max_tanimoto_vs_train_morgan'),
        ('nearest neighbour, confirmed train-split subset',
         'max_tanimoto_vs_trainsplit_confirmed_rdkit', 'max_tanimoto_vs_trainsplit_confirmed_morgan'),
        ('caffeine, the pocket reference ligand',
         'tanimoto_vs_caffeine_rdkit', 'tanimoto_vs_caffeine_morgan')]:
    print(f'{label:<48s}{nov[rd].median():>18.4f}{nov[mg].median():>13.4f}')

r_rd = nov['max_tanimoto_vs_train_rdkit'].median() / nov['tanimoto_vs_caffeine_rdkit'].median()
r_mg = nov['max_tanimoto_vs_train_morgan'].median() / nov['tanimoto_vs_caffeine_morgan'].median()
print(f'\n{"training corpus / caffeine ratio":<48s}{r_rd:>17.2f}x{r_mg:>12.2f}x')

The gap is the finding. Under the path-based fingerprint, the fingerprint that produced the published
0.21, these ligands sit at median 0.2121 Tanimoto to caffeine and at median 0.4664 to their nearest
neighbour in a sample of the training corpus. That is 2.20x higher. Under Morgan r=2 the same comparison
runs 0.0899 against caffeine and 0.2144 against the corpus, a ratio of 2.38x. Restricting the corpus to
the 1,749 ligands independently confirmed present in DiffSBDD's own published train split gives 0.4453,
which is close enough to the full-sample figure that the two readings agree.

Median-of-maximum around 0.45 is not memorization. Identical molecules score 1.0, and a medicinal chemist
would call 0.45 on a path-based fingerprint a recognizable family resemblance rather than a copy. But it
is more than twice the number the chapter prints as novelty, and it is the number that speaks to the
question readers are asking. Both belong in the write-up, each under its own name: dissimilarity to the
reference ligand, and proximity to the training distribution.

In [ ]:
# How much does the answer depend on how much corpus we searched? A nearest-neighbour
# similarity can only RISE as the corpus grows, so this curve tells us whether the number above
# is close to its limit or merely the best we could see. Nested prefixes of one seeded shuffle,
# so each corpus is a superset of the one before and the comparison is apples to apples.
import numpy as np

corpus_smiles, in_train_split = read_corpus_smi(CORPUS_SMI)
corpus_rdkit = [Chem.RDKFingerprint(m) for m in
                (Chem.MolFromSmiles(s) for s in corpus_smiles) if m is not None]
gen_rdkit = [Chem.RDKFingerprint(m) for m in
             (Chem.MolFromSmiles(s) for s in nov['smiles']) if m is not None]

order = np.random.default_rng(4242).permutation(len(corpus_rdkit))
print(f'{"corpus size":>12s}{"median-of-max Tanimoto":>26s}')
for divisor in (8, 4, 2, 1):
    n = len(order) // divisor
    subset = [corpus_rdkit[k] for k in order[:n]]
    medians = [max(DataStructs.BulkTanimotoSimilarity(fp, subset)) for fp in gen_rdkit]
    print(f'{n:>12d}{float(np.median(medians)):>26.4f}')

The curve rises the whole way and never flattens: 0.4423 at 651 corpus molecules, 0.4449 at 1,302,
0.4488 at 2,605, and 0.4664 at the full 5,211-molecule sample. So **every training-proximity number in
this notebook is a lower bound.** The official train split names 97,030 distinct ligand files, and our
5,211-molecule sample confirms only 1,749 of them, so we searched under 2% of what the model was trained
on; adding molecules can only find closer neighbours, never further ones. True similarity to the training
distribution is higher than 0.4664. That strengthens the point rather than weakening it, and it is why we
report the figure as a floor instead of an estimate.

The corpus provenance deserves the same directness. DiffSBDD's released checkpoint was trained on
CrossDocked2020, but the exact training-split files behind it are not published as a SMILES list, so no
one can compute this number against precisely what the model saw. Our corpus is a documented public
preprocessing of CrossDocked2020, the `kohbanye/crossdocked2020` dataset on HuggingFace (CC0), sampled by
seeded HTTP range reads over 33 of its 39 archive shards and deduplicated to 5,211 unique ligands by
canonical SMILES. That is a proxy, and we label it as one. The hedge-free version of the number is the
confirmed-subset figure: 1,749 of those ligands were matched by name against the official
`split_by_name.pt` train list, so they are known to be training molecules, and they give 0.4453. Two
routes, one with a caveat and one without, landing in the same place.

In [ ]:
# Diversity and uniqueness, with the definitions in the code rather than in a footnote.
#   diversity  = 1 - mean pairwise Tanimoto over the valid molecules
#   uniqueness = distinct canonical SMILES / number of valid molecules
# Computed under both fingerprints, because the fingerprint changes the answer materially.
from itertools import combinations
import numpy as np

heavy_mols = [Chem.RemoveHs(m) for m in valid_mols]


def diversity_and_uniqueness(mols, fingerprint):
    fps = [fingerprint(m) for m in mols]
    pairs = list(combinations(range(len(fps)), 2))
    sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j]) for i, j in pairs]
    # np.mean, not sum()/len(): pairwise summation changes the last two digits of the float,
    # and we want the published value back exactly rather than to 15 places.
    diversity = 1 - float(np.mean(sims))
    n_distinct = len({Chem.MolToSmiles(m) for m in mols})
    return diversity, len(pairs), n_distinct


for name, fingerprint in [('RDKit path-based (fpSize=2048, minPath=1, maxPath=7, nBitsPerHash=2)',
                           Chem.RDKFingerprint),
                          ('Morgan circular (radius=2, fpSize=2048)',
                           morgan_gen.GetFingerprint)]:
    div, n_pairs, n_distinct = diversity_and_uniqueness(heavy_mols, fingerprint)
    print(f'{name}')
    print(f'  diversity  = {div:.16f}')
    print(f'               over {n_pairs} pairs of {len(heavy_mols)} molecules')
    print(f'  uniqueness = {n_distinct/len(heavy_mols):.2f}'
          f'   ({n_distinct} distinct canonical SMILES)')

The published 0.809 reproduces bit-for-bit: 0.8089776623453144 under `Chem.RDKFingerprint` at RDKit's
defaults, over the 20 valid molecules, with diversity defined as one minus the mean pairwise Tanimoto over
all 190 pairs. That pins the number's provenance, which was the goal.

It also shows why the definition has to travel with the number. The **same 20 molecules** score
**0.9175** under Morgan r=2/2048. The fingerprint moved the answer by 0.11, which is larger than the
difference most papers report between models, and Morgan is what a reader would probably assume if you
said "diversity" without qualification. Neither value is wrong and the higher one is not better; they are
answers to different questions about the same batch. Quote diversity the way this cell prints it, with
the fingerprint, its parameters and the formula, or the number cannot be checked.

Uniqueness is 1.00 under either fingerprint, since it does not depend on one: all 20 canonical SMILES are
distinct, so the model is not quietly returning the same molecule twenty times. For orientation,
FlowMol3's unconditioned batch in Notebook 3 scores 0.7894 path-based and 0.8695 Morgan on the same
definitions, also at 1.00 uniqueness.

## <b> <font color='#A20025'> See one ligand in the pocket</font> </b>

The whole promise of SBDD is *placement*; the ligand is generated already sitting in the binding site, in the pocket's coordinate frame. Let's look. We take the **highest-QED** generated ligand and render it inside the pocket residues with **py3Dmol** (interactive in Colab/Jupyter; rotate it with the mouse).

> If you are reading a static export, the chapter also saves a rendered snapshot (`diffsbdd_pocket_view.png`) and the full complex as `diffsbdd_complex.pdb`.

In [ ]:
import py3Dmol
best = max(valid_mols, key=lambda m: QED.qed(m))
best_idx = int(np.argmax(qed))
print('best ligand:', Chem.MolToSmiles(best), '| QED', round(QED.qed(best),2))

# pocket residues (protein atoms within 8A of the reference ligand)
from Bio.PDB import PDBParser, PDBIO, Select
pdb_model = PDBParser(QUIET=True).get_structure('x', PDB)[0]
pocket_res = utils.get_pocket_from_ligand(pdb_model, REF)
resset = set((r.get_parent().id, r.id[1]) for r in pocket_res)
class _Sel(Select):
    def accept_residue(self, r): return (r.get_parent().id, r.id[1]) in resset
io = PDBIO(); io.set_structure(pdb_model); io.save('_pocket.pdb', _Sel())
pocket_pdb = open('_pocket.pdb').read()
lig_pdb = Chem.MolToPDBBlock(raw_mols[best_idx])

view = py3Dmol.view(width=560, height=420)
view.addModel(pocket_pdb, 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'spectrum'},
                             'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.08}})
view.addModel(lig_pdb, 'pdb')
view.setStyle({'model': 1}, {'stick': {'colorscheme': 'redCarbon', 'radius': 0.18}})
view.zoomTo({'model': 1})
view.show()

![Figure I.10]({{artifact:2b0d3325-fd00-4a03-8055-23c51cda078a}})

**Figure I.10.** The highest-QED generated ligand seated in the A2A binding pocket. The ligand is drawn as sticks colored by element (blue nitrogen, red oxygen, yellow sulfur, dark carbon) nestled inside a translucent surface traced from the 3RFM pocket residues. This is a single de novo molecule that the pocket-conditioned diffusion model placed directly into the binding site, with a drug-like QED of 0.79.

*The `view.show()` cell above is interactive in Colab/Jupyter; drag to rotate the ligand inside its pocket. The chapter also saves a static snapshot (`diffsbdd_pocket_view.png`) and the full complex (`diffsbdd_complex.pdb`, which opens in any molecular viewer). The highest-QED ligand here; `O=C(NC1=CCCCC1)N1SCc2scnc21`, QED 0.79; was never told these coordinates; the fixed pocket context placed it there through the reverse diffusion.*

## <b> <font color='#A20025'> One checkpoint, many tasks: optimize a molecule</font> </b>

Here is the payoff that makes DiffSBDD more than a one-trick generator. The **same checkpoint** you just used for de-novo design also runs **property optimization**; no retraining, no second model.

The idea is a tiny **evolutionary loop** built on a *noise-then-denoise* move. Take a molecule, add noise for only a *partial* number of steps (so you don't destroy it entirely), then denoise it back inside the pocket. You get a **diversified variant**; close to the original but not identical. Score every variant by the objective (here **QED**), keep the top-$k$, and repeat. Over a few generations the population drifts toward higher-scoring molecules.

We seed the loop with **caffeine itself** and optimize for QED, at a deliberately small CPU scale.

> **A note on scale.** The DiffSBDD README runs this with `population_size 100`, `evolution_steps 10`, `top_k 7`, `timesteps 100`. That is far too heavy for a CPU. We first planned `population_size 10, evolution_steps 4`, then trimmed once more to **`population_size 8, evolution_steps 3, top_k 3, timesteps 50`**; the smallest setting that still shows a clear QED climb in a few minutes on CPU. The recipe is identical at any scale; only the counts change. On a GPU, restore the README values for a stronger search.

In [ ]:
# Runs the repo's optimize.py. Small scale so it finishes on CPU in a few minutes.
# (In the chapter: population 8, 3 generations, top-3, 50 timesteps.)
!python optimize.py --checkpoint {CKPT} --pdbfile {PDB} \
    --ref_ligand example/3rfm_B_CFF.sdf --objective qed \
    --population_size 8 --evolution_steps 3 --top_k 3 --timesteps 50 \
    --outfile diffsbdd_optimized.sdf

# NOTE: on pandas>=2.0 the repo's optimize.py needs a one-line compat shim
# (DataFrame.append was removed). See the chapter's patched copy; the fix is:
#   if not hasattr(pd.DataFrame,'append'):
#       pd.DataFrame.append = lambda self,o,ignore_index=False: pd.concat(
#           [self, pd.DataFrame([o])], ignore_index=ignore_index)

![Figure I.8]({{artifact:d6784eb1-d514-4186-a5cd-1e55216b212c}})

**Figure I.8.** Three generations of DiffSBDD's evolutionary optimization steadily raise ligand quality. Starting from a seed molecule with QED 0.54, the best-in-population QED climbs generation by generation to 0.72, a gain of nearly 0.18 in just three rounds. The optimization reuses the same pretrained checkpoint, showing that one model supports both fresh generation and targeted property improvement.

`optimize.py` writes its full population history to a CSV buffer alongside the SDF. Let's read it back and plot the QED trajectory; the clearest picture of the optimization at work.

In [ ]:
import json, pandas as pd

# The optimization trajectory ships as JSON (mean and max QED per generation, plus the seed),
# which is exactly what this plot needs. Set USE_CACHED = False only if you re-ran optimize.py
# above and want to read its per-molecule CSV instead.
USE_CACHED = True

if USE_CACHED:
    metrics_path = DATA / 'diffsbdd_optimize_metrics.json'
    if not metrics_path.exists():
        metrics_path = 'diffsbdd_optimize_metrics.json'
    opt = json.load(open(metrics_path))
    traj = pd.DataFrame([{'generation': g['generation'], 'mean': g['mean_qed'], 'max': g['max_qed']}
                         for g in opt['trajectory']])
    seed_qed = float(opt['seed_qed'])
else:
    buf = pd.read_csv('diffsbdd_optimized.csv')      # written by optimize.py, next to the SDF
    traj = buf.groupby('generation')['score'].agg(['mean', 'max']).reset_index()
    seed_qed = float(buf[buf.generation == 0]['score'].iloc[0])

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.plot(traj['generation'], traj['max'],  '-o', color='#A20025', lw=2.2, ms=7, label='best in population')
ax.plot(traj['generation'], traj['mean'], '-s', color='#c98a97', lw=1.8, ms=6, label='population mean')
ax.axhline(seed_qed, color='#333', ls='--', lw=1.2)
ax.annotate(f'caffeine seed (QED {seed_qed:.2f})', xy=(0.05, seed_qed), xytext=(0.05, seed_qed-0.03), fontsize=8, style='italic')
ax.set_xlabel('evolution generation'); ax.set_ylabel('QED (drug-likeness)')
ax.set_xticks(traj['generation']); ax.set_title('QED optimization from caffeine; one checkpoint, no retraining')
ax.legend(frameon=False); fig.tight_layout(); plt.show()
print('best optimized QED:', round(buf['score'].max(), 3), '| best SMILES:', buf.loc[buf['score'].idxmax(),'smiles'])

The QED climbs generation over generation; from caffeine's 0.54 to a best of **0.72** in three short generations, all from the one checkpoint you already loaded. In the chapter run the population history was:

| generation | population mean QED | best in population |
|---:|---:|---:|
| 0 (caffeine seed) | 0.54 | 0.54 |
| 1 | 0.56 | 0.59 |
| 2 | 0.58 | 0.63 |
| 3 | 0.63 | **0.72** |

Both the best-in-population and the population mean rise monotonically above the caffeine seed. Same weights as §3; the only difference is the noise-then-denoise loop wrapped around them. Output molecules are in `diffsbdd_optimized.sdf`.

> **Why this matters.** A classic generative pipeline needs a separate model (or a separate fine-tune) for every task: one to generate, another to optimize, another to link fragments. DiffSBDD's diffusion formulation gives you generation, optimization, and substructure **inpainting** (scaffold hopping, fragment linking; see `inpaint.py`) from a *single* set of weights. That is the structural advantage of diffusion for drug design.

## <b> <font color='#A20025'> Keeping a scaffold: inpainting</font> </b>

De novo generation and optimization both let DiffSBDD redraw the *whole* ligand. But medicinal chemists often want the opposite: keep a specific substructure; a scaffold that's known to bind, patent-free, or synthetically convenient; and regenerate only the **rest** of the molecule around it. Diffusion models do this with **inpainting**, exactly as image models fill in a masked region of a photo while leaving the rest untouched.

Here the "unmasked" region is a set of atoms whose positions and identities we pin. The model diffuses new atoms around them, re-harmonizing the new chemistry with the fixed core at every reverse step (that's the `resamplings` knob). We'll fix the **purine core of caffeine**; the fused two-ring system, nine atoms; and let DiffSBDD grow the rest inside the A₂A pocket.

In [ ]:
# Build the fixed scaffold: the 9-atom purine core of the caffeine reference ligand,
# keeping its 3-D coordinates. inpaint.py reads this SDF with sanitize=False and uses
# only the atom positions + elements, so we write it unkekulized.
from rdkit import Chem
from rdkit.Chem import RWMol

ref = Chem.SDMolSupplier('example/3rfm_B_CFF.sdf', sanitize=False)[0]
keep = [0,1,3,4,7,8,9,11,12]          # the fused purine ring atoms
em = RWMol(ref)
for idx in sorted([a.GetIdx() for a in ref.GetAtoms() if a.GetIdx() not in keep], reverse=True):
    em.RemoveAtom(idx)
w = Chem.SDWriter('scaffold_caffeine_core.sdf'); w.SetKekulize(False)
w.write(em.GetMol()); w.close()
print('fixed scaffold: 9 purine-core atoms with 3-D coordinates')

![Figure I.9]({{artifact:127fb0b2-cfb2-42ea-91ef-490da0667847}})

**Figure I.9.** Scaffold-constrained inpainting fixes the caffeine purine core and regenerates the rest. The fixed scaffold atoms are held in place while DiffSBDD grows new chemistry around them, so every one of the ten valid results retains the exact core geometry while varying the substituents. This is the third distinct job the same checkpoint performs: de novo generation, property optimization, and now scaffold-constrained inpainting.

One repo quirk to know about: `inpaint.py` references a module-global `model` inside a helper, which only exists when you run it as a script. Calling it as a script (below) is the clean path. The chapter's driver imports the function instead and sets `inpaint.model = model` first; either works.

In [ ]:
# Run inpainting as a script (avoids the module-global gotcha). resamplings is the
# inpainting-specific knob: at each timestep the model re-harmonizes generated atoms
# with the fixed ones this many times. Chapter run: 10 samples, 50 steps, 8 resamplings
# (~23 min on CPU). Load the cached SDF to skip the wait (see USE_CACHED below).
USE_CACHED = True   # flip to False to actually run inpainting

if USE_CACHED:
    from rdkit import Chem
    inpainted = [m for m in Chem.SDMolSupplier(str(DATA / "diffsbdd_inpainted.sdf"), sanitize=False) if m]
    print(f'loaded {len(inpainted)} cached inpainted molecules')
else:
    !python inpaint.py {CKPT} --pdbfile {PDB} \
        --ref_ligand A:330 --fix_atoms scaffold_caffeine_core.sdf \
        --n_samples 10 --timesteps 50 --resamplings 8 \
        --outfile diffsbdd_inpainted.sdf
    from rdkit import Chem
    inpainted = [m for m in Chem.SDMolSupplier('diffsbdd_inpainted.sdf', sanitize=False) if m]

**What we found; and an honest subtlety.** All 10 inpainted molecules were valid (median QED 0.56, right around caffeine's 0.54; best 0.76). The number that matters for inpainting is *scaffold retention*, and by the ground-truth measure; do the nine fixed atoms stay in their pinned positions?; retention was **1.00 for all ten**. But when we checked with a strict aromatic-purine SMARTS pattern instead of coordinates, only 2/10 matched. The reason is worth knowing before it surprises you: inpainting fixes atom *positions and elements*, but bond orders are re-perceived from geometry afterward (here by OpenBabel), and that perception can assign a slightly different aromatic/aliphatic pattern than the original. **The atoms are held; the drawn bonds may vary.** Figure I.9 in the chapter shows the QED distribution and six of the elaborated structures.

In [ ]:
# Score the inpainted set the same way as the de novo batch.
import numpy as np
from rdkit.Chem import QED
qeds = []
for m in inpainted:
    try:
        mm = Chem.Mol(m); Chem.SanitizeMol(mm); qeds.append(QED.qed(mm))
    except Exception:
        pass
print(f'valid: {len(qeds)}/{len(inpainted)}  |  median QED {np.median(qeds):.2f}  |  best {max(qeds):.2f}')
print('caffeine QED 0.54  |  scaffold coordinate retention 1.00 (all 10)')

![Figure I.11]({{artifact:f1c6e31d-33de-42a7-bc46-47913a0689e2}})

**Figure I.11.** The two directions of structure-based design are mirror images of the same denoiser. On one side the pocket is fixed and the ligand is generated; on the other the ligand is fixed and the pocket is designed around it. Both use the identical equivariant diffusion process run with different atoms held constant, which is why a single trained model spans the whole design space.

## <b> <font color='#A20025'> Summary</font> </b>

You ran the chapter's structure-based capstone entirely from a pretrained checkpoint, on CPU:

- **Pocket-conditioned diffusion.** DiffSBDD noises and denoises **only the ligand**, always conditioned on a **fixed protein pocket**, using an **SE(3)-equivariant** network so 3D binding geometry is learned without memorising every orientation. The pocket is a mold; noise flows into a ligand that fits it.
- **De-novo generation.** From the adenosine A$_{2A}$ receptor pocket (`3RFM`, defined by bound caffeine) you generated valid 3D ligands; **100% graph validity** in the chapter run; written to SDF with pocket coordinates. On the physical bar the same batch is **15 of 20** PoseBusters-valid against the full receptor, with every failure intramolecular and every ligand-to-protein check passing, so the model places molecules correctly while straining their internal geometry: median UFF strain **134.48 kcal/mol** against the co-crystal reference's **0.388**.
- **Profiling.** The generated set was **as drug-like as caffeine or better** (median QED 0.60), **synthesizable**, and **far from the reference ligand** (every molecule below 0.4 Tanimoto to caffeine, median 0.2121). Measured against a sample of its training corpus instead, median nearest-neighbour similarity is **0.4664**, 2.20x higher, so the caffeine figure is distance from one reference and not evidence about memorisation. Pairwise diversity **0.8090** reproduces exactly under `Chem.RDKFingerprint` at RDKit defaults and is **0.9175** under Morgan r=2, so the fingerprint has to be named for the number to mean anything. Details in `diffsbdd_metrics.csv`, `novelty_diffsbdd.csv` and `diversity_metadata.json`.
- **One checkpoint, many tasks.** The same weights optimized a molecule for QED via a noise-then-denoise evolutionary loop; caffeine **0.54 → 0.72** in three generations, no retraining; *and* inpainted new chemistry around a **fixed purine scaffold** (10/10 valid, coordinate retention 1.00). De novo, optimization, and inpainting: three jobs, one checkpoint.

**The two directions of structure-based design (bridge to §I.6).** DiffSBDD generates a *ligand* given a *fixed pocket*. The mirror image; fixing a *ligand* and generating or refolding the *protein* around it; is what protein-centric diffusion models like RFdiffusion do. Same diffusion machinery, opposite conditioning. **Next (§I.7):** you take one generated ligand and hand it to a structure-prediction/docking connector (Boltz) to score it against the real target; closing the loop from noise to a testable hypothesis.

### <b> <font color='#A20025'> References</font> </b>

1. Schneuing, A., Harris, C., Du, Y., Didi, K., Jamasb, A., Igashov, I., Du, W., Gomes, C., Blundell, T. L., Lió, P., Welling, M., Bronstein, M. & Correia, B. (2024), *Structure-based drug design with equivariant diffusion models*, **Nature Computational Science** 4, 899-909. doi:10.1038/s43588-024-00737-x (arXiv:2210.13695). Code (MIT): https://github.com/arneschneuing/DiffSBDD
2. Pretrained weights: Zenodo record 8183747, https://zenodo.org/record/8183747 (`crossdocked_fullatom_cond.ckpt`). Trained on **CrossDocked2020** (GPL-2.0); dataset **not** redistributed here.
3. Satorras, V. G., Hoogeboom, E. & Welling, M. (2021), *E(n) Equivariant Graph Neural Networks*, ICML. arXiv:2102.09844. *(the EGNN backbone)*
4. Hoogeboom, E., Satorras, V. G., Vignac, C. & Welling, M. (2022), *Equivariant Diffusion for Molecule Generation in 3D*, ICML. arXiv:2203.17003. *(EDM; the diffusion formulation DiffSBDD builds on)*
5. Doré, A. S. et al. (2011), *Structure of the adenosine A$_{2A}$ receptor in complex with... caffeine*, **Structure** 19, 1283-1293. *(PDB 3RFM)*
6. Bickerton, G. R. et al. (2012), *Quantifying the chemical beauty of drugs*, **Nat. Chem.** *(QED)*. Ertl, P. & Schuffenhauer, A. (2009), *J. Cheminform.* *(SA score)*.
7. Watson, J. L. et al. (2023), *De novo design of protein structure and function with RFdiffusion*, **Nature** 620, 1089-1100. *(the protein-centric mirror image, §I.6)*
8. Buttenschoen, M., Morris, G. M. & Deane, C. M. (2024), *PoseBusters: AI-based docking methods fail to generate physically valid poses or generalise to novel sequences*, **Chemical Science** 15, 3130-3139. Code (BSD-3): https://github.com/maabuu/posebusters
9. Harris, C., Didi, K., Jamasb, A. R., Joshi, C. K., Mathis, S. V., Lio, P. & Blundell, T. (2023), *Benchmarking Generated Poses: How Rational is Structure-based Drug Design with Generative Models?* arXiv:2308.07413. *(PoseCheck: strain energy, clashes and interaction profiles)*